# ResolvFL sobre ImageNet-3L/F: protocolo auditado y violación de resolvability

Este notebook reproduce la comparación principal sobre ImageNet-3L/F y añade
el experimento de **violación controlada de resolvability** en un nodo
`subordinate`. El objetivo de la intervención es eliminar únicamente la
supervisión que distingue las clases `fine` descendientes de ese nodo,
manteniendo intacta la supervisión de su prefijo `basic -> subordinate`.

La versión está preparada para Google Colab y mantiene una explicación
detallada de cada decisión experimental. Se separan claramente:

1. la reconstrucción y validación de la taxonomía;
2. la partición doble-Dirichlet;
3. el auditoría de cobertura empírica;
4. el entrenamiento principal;
5. la selección determinista del nodo intervenido;
6. el entrenamiento emparejado resoluble/violado;
7. las métricas dentro del subárbol, fuera de él y en el prefijo resuelto.

## Correcciones introducidas durante la auditoría

- La pérdida jerárquica se implementa como la **media de la suma de pérdidas
  a lo largo del camino de cada muestra**. Para ello, cada término usa
  `reduction="sum"` y la suma total se divide por el tamaño del minibatch.
  Sumar medias independientes por nodo sobreponderaría los nodos con pocas
  muestras y no coincidiría con la likelihood marginalizada descrita en el
  artículo.
- Las permutaciones de minibatches usan un generador local con semilla
  explícita y su estado se guarda en los checkpoints.
- Los checkpoints quedan aislados mediante una versión de protocolo y se
  validan contra un manifiesto antes de reanudarse.
- La representación DINO permanece completamente congelada. Este notebook
  entrena únicamente las cabezas lineales; **no incluye una proyección
  compartida entrenable**.
- El modo rápido limita rondas y muestras por cliente, pero sus resultados
  son solo una prueba de humo y nunca deben incorporarse al artículo.

**Importante:** estas correcciones cambian el protocolo de optimización
respecto a cualquier ejecución previa que sumara medias por nodo. Los
checkpoints antiguos no se reutilizan y los resultados destinados al artículo
deben obtenerse con esta versión auditada.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 1. Dependencias y configuración

Los interruptores `RUN_MAIN_EXPERIMENT` y
`RUN_RESOLVABILITY_VIOLATION` permiten ejecutar por separado la tabla principal
y la intervención. La violación necesita dos entrenamientos de ResolvFL por
semilla: uno resoluble y otro con la supervisión fina del nodo seleccionado
eliminada.

La versión del protocolo forma parte de todos los nombres de checkpoints.
Cambiar hiperparámetros relevantes exige cambiar también
`EXPERIMENT_VERSION`; así se evita reanudar silenciosamente una ejecución
incompatible.


In [ ]:
import copy
import hashlib
import json
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------------
# RUTAS Y MODO DE EJECUCIÓN
# ---------------------------------------------------------------------
FEAT_DIR = Path("/content/drive/MyDrive/features_dino")
BASE_CKPT_DIR = Path("/content/drive/MyDrive/imagenet3lf_ckpt")
EXPERIMENT_VERSION = "v4_exact_pathloss_violation"

# ---------------------------------------------------------------------
# QUÉ EJECUTAR
# ---------------------------------------------------------------------
QUICK_TEST = False
RUN_MAIN_EXPERIMENT = False
RUN_RESOLVABILITY_VIOLATION = True
ALLOW_RESUME = True

# El modo rápido sirve solo para verificar que el código completa el flujo.
QUICK_ROUNDS = 2
QUICK_MAX_SAMPLES_PER_CLIENT = 2048
QUICK_MAX_VAL_SAMPLES = 4096

def get_ckpt_dir():
    """Devuelve una carpeta distinta para quick test y ejecución completa."""
    mode = "quick" if QUICK_TEST else "full"
    path = BASE_CKPT_DIR / f"{EXPERIMENT_VERSION}_{mode}"
    path.mkdir(parents=True, exist_ok=True)
    return path

CKPT_DIR = get_ckpt_dir()

# ---------------------------------------------------------------------
# PROTOCOLO PRINCIPAL
# ---------------------------------------------------------------------
SEEDS = [1, 2, 3]
K = 10
ALPHA_CLS = 0.5
ALPHA_DEP = 0.5
N_ROUNDS = 150
EVAL_EVERY = 10
LOCAL_EPOCHS = 1
LR = 0.05
BATCH = 512

# Se conservan las semillas 1/2/3 de la partición original, pero cada fuente
# de aleatoriedad queda identificada por separado.
PARTITION_SEED = {s: s for s in SEEDS}
INIT_SEED = {s: 20_000 + s for s in SEEDS}
ORDER_SEED = {s: 30_000 + s for s in SEEDS}

# ---------------------------------------------------------------------
# SELECCIÓN DEL NODO PARA LA VIOLACIÓN
# ---------------------------------------------------------------------
# Se prioriza un subordinate con cinco hojas, como en las intervenciones de
# CIFAR-100 y del subconjunto Aves de iNat2021-mini. La selección se basa solo
# en estructura y soporte, nunca en el rendimiento del modelo.
TARGET_SUBORD_OVERRIDE = None
PREFERRED_TARGET_LEAVES = 5
FALLBACK_MIN_LEAVES = 4
FALLBACK_MAX_LEAVES = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("checkpoint directory:", get_ckpt_dir())


In [ ]:
# Diagnóstico obligatorio: quick y full nunca comparten carpeta.
print("QUICK_TEST =", QUICK_TEST)
print("EXPERIMENT_VERSION =", EXPERIMENT_VERSION)
print("ACTIVE_CKPT_DIR =", get_ckpt_dir())
expected_suffix = "_quick" if QUICK_TEST else "_full"
assert get_ckpt_dir().name.endswith(expected_suffix)


## 2. Utilidades de reproducibilidad y escritura segura

Los ficheros JSON y los checkpoints se escriben primero en un temporal y se
renombran al final. De este modo, una desconexión de Colab no deja un fichero
parcial con apariencia válida.

El manifiesto de cada ejecución registra la partición, las etiquetas
observadas, la inicialización y los hiperparámetros. Un checkpoint solo puede
reanudarse si su manifiesto coincide exactamente con el protocolo solicitado.


In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def atomic_json_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True)
    os.replace(tmp, path)


def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def array_sha256(arr):
    arr = np.ascontiguousarray(arr)
    h = hashlib.sha256()
    h.update(str(arr.dtype).encode("utf-8"))
    h.update(str(arr.shape).encode("utf-8"))
    h.update(arr.tobytes())
    return h.hexdigest()


def state_dict_sha256(state_dict):
    h = hashlib.sha256()
    for key in sorted(state_dict):
        tensor = state_dict[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tensor.dtype).encode("utf-8"))
        h.update(str(tuple(tensor.shape)).encode("utf-8"))
        h.update(tensor.numpy().tobytes())
    return h.hexdigest()


def assert_manifest_matches(saved, expected):
    if saved != expected:
        saved_text = json.dumps(saved, indent=2, sort_keys=True)
        expected_text = json.dumps(expected, indent=2, sort_keys=True)
        raise RuntimeError(
            "El checkpoint existe, pero su manifiesto no coincide con el "
            "protocolo actual.\n\nGUARDADO:\n"
            + saved_text
            + "\n\nESPERADO:\n"
            + expected_text
        )


def load_torch_checkpoint(path, map_location):
    # Compatibilidad con versiones de PyTorch anteriores y posteriores al
    # cambio del argumento weights_only.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def mean_std(values):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    std = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    return mean, std


## 3. Carga de features y etiquetas

Las features se mantienen como `memmap` y se convierten a tensor únicamente
por minibatches. Esto evita copiar en RAM todo el fichero de entrenamiento,
que puede ocupar varios gigabytes.

Se espera que cada fila de etiquetas contenga los identificadores
`basic`, `subordinate` y `fine`.


In [ ]:
required_files = {
    "train_features": FEAT_DIR / "train_features.npy",
    "train_labels": FEAT_DIR / "train_labels.npy",
    "val_features": FEAT_DIR / "val_features.npy",
    "val_labels": FEAT_DIR / "val_labels.npy",
}
missing = [str(p) for p in required_files.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan los siguientes ficheros de features/etiquetas:\n"
        + "\n".join(missing)
    )

Xtr = np.load(required_files["train_features"], mmap_mode="r")
Ltr = np.load(required_files["train_labels"])
Xva = np.load(required_files["val_features"], mmap_mode="r")
Lva = np.load(required_files["val_labels"])

assert Xtr.ndim == 2 and Xva.ndim == 2
assert Ltr.ndim == 2 and Ltr.shape[1] >= 3
assert Lva.ndim == 2 and Lva.shape[1] >= 3
assert len(Xtr) == len(Ltr)
assert len(Xva) == len(Lva)
assert Xtr.shape[1] == Xva.shape[1]

FEATURE_DIM = int(Xtr.shape[1])
print(f"train: features={Xtr.shape}, labels={Ltr.shape}")
print(f"val:   features={Xva.shape}, labels={Lva.shape}")
print("feature dimension:", FEATURE_DIM)


## 4. Reconstrucción y validación de la taxonomía

La taxonomía se reconstruye a partir de las triples observadas en
entrenamiento. No se supone que los identificadores originales sean
consecutivos ni que coincidan con los índices locales usados por los
clasificadores.

Las comprobaciones siguientes garantizan que:

- cada `subordinate` tiene un único padre `basic`;
- cada clase `fine` tiene un único padre `subordinate`;
- todas las etiquetas de validación existen en entrenamiento;
- la estructura contiene exactamente 20 nodos `basic`, 127 `subordinate` y
  505 hojas, como exige ImageNet-3L/F.


In [ ]:
def py_scalar(x):
    return x.item() if hasattr(x, "item") else x


# ----------------------------- BASIC ---------------------------------
basic_values = sorted({py_scalar(x) for x in Ltr[:, 0]})
basic_raw_to_local = {raw: i for i, raw in enumerate(basic_values)}
basic_local = np.array(
    [basic_raw_to_local[py_scalar(x)] for x in Ltr[:, 0]],
    dtype=np.int64,
)

# --------------------------- SUBORDINATE ------------------------------
subord_key_to_local = {}
subord_parent_list = []
subord_raw_list = []
subord_local = np.empty(len(Ltr), dtype=np.int64)

for i, (b_raw, s_raw) in enumerate(zip(Ltr[:, 0], Ltr[:, 1])):
    b = basic_raw_to_local[py_scalar(b_raw)]
    key = (b, py_scalar(s_raw))
    if key not in subord_key_to_local:
        subord_key_to_local[key] = len(subord_key_to_local)
        subord_parent_list.append(b)
        subord_raw_list.append(py_scalar(s_raw))
    subord_local[i] = subord_key_to_local[key]

subord_parent = np.asarray(subord_parent_list, dtype=np.int64)

# ------------------------------- LEAF --------------------------------
leaf_key_to_local = {}
leaf_parent_list = []
leaf_raw_list = []
leaf_local = np.empty(len(Ltr), dtype=np.int64)

for i, (s, l_raw) in enumerate(zip(subord_local, Ltr[:, 2])):
    key = (int(s), py_scalar(l_raw))
    if key not in leaf_key_to_local:
        leaf_key_to_local[key] = len(leaf_key_to_local)
        leaf_parent_list.append(int(s))
        leaf_raw_list.append(py_scalar(l_raw))
    leaf_local[i] = leaf_key_to_local[key]

leaf_parent_local = np.asarray(leaf_parent_list, dtype=np.int64)

N_BASIC = len(basic_values)
N_SUBORD = len(subord_parent)
N_LEAF = len(leaf_parent_local)

assert (N_BASIC, N_SUBORD, N_LEAF) == (20, 127, 505), (
    "La taxonomía reconstruida no coincide con ImageNet-3L/F: "
    f"{N_BASIC}/{N_SUBORD}/{N_LEAF}"
)

children_per_basic = np.bincount(subord_parent, minlength=N_BASIC)
children_per_subord = np.bincount(leaf_parent_local, minlength=N_SUBORD)
assert children_per_basic.min() > 0
assert children_per_subord.min() > 0

subords_by_basic = [
    np.where(subord_parent == b)[0].astype(np.int64)
    for b in range(N_BASIC)
]
leaves_by_subord = [
    np.where(leaf_parent_local == s)[0].astype(np.int64)
    for s in range(N_SUBORD)
]

basic_child_index = {
    b: {int(s): j for j, s in enumerate(subords_by_basic[b])}
    for b in range(N_BASIC)
}
subord_child_index = {
    s: {int(l): j for j, l in enumerate(leaves_by_subord[s])}
    for s in range(N_SUBORD)
}

# -------------------------- VALIDACIÓN -------------------------------
val_basic_local = np.empty(len(Lva), dtype=np.int64)
val_subord_local = np.empty(len(Lva), dtype=np.int64)
val_leaf_local = np.empty(len(Lva), dtype=np.int64)

for i, (b_raw, s_raw, l_raw) in enumerate(Lva[:, :3]):
    b_key = py_scalar(b_raw)
    if b_key not in basic_raw_to_local:
        raise KeyError(f"Basic de validación no visto en train: {b_key}")
    b = basic_raw_to_local[b_key]

    s_key = (b, py_scalar(s_raw))
    if s_key not in subord_key_to_local:
        raise KeyError(f"Subordinate de validación no visto en train: {s_key}")
    s = subord_key_to_local[s_key]

    l_key = (s, py_scalar(l_raw))
    if l_key not in leaf_key_to_local:
        raise KeyError(f"Fine class de validación no vista en train: {l_key}")
    l = leaf_key_to_local[l_key]

    val_basic_local[i] = b
    val_subord_local[i] = s
    val_leaf_local[i] = l

assert np.array_equal(subord_parent[val_subord_local], val_basic_local)
assert np.array_equal(leaf_parent_local[val_leaf_local], val_subord_local)

def local_targets(basic_arr, subord_arr, leaf_arr):
    t_root = np.asarray(basic_arr, dtype=np.int64)
    t_basic = np.array(
        [basic_child_index[int(b)][int(s)] for b, s in zip(basic_arr, subord_arr)],
        dtype=np.int64,
    )
    t_subord = np.array(
        [subord_child_index[int(s)][int(l)] for s, l in zip(subord_arr, leaf_arr)],
        dtype=np.int64,
    )
    return t_root, t_basic, t_subord

T_ROOT_ALL, T_BASIC_ALL, T_SUBORD_ALL = local_targets(
    basic_local, subord_local, leaf_local
)

# Espacio plano: basic + subordinate + fine.
FLAT_OFFSET_BASIC = 0
FLAT_OFFSET_SUBORD = N_BASIC
FLAT_OFFSET_LEAF = N_BASIC + N_SUBORD
N_FLAT = N_BASIC + N_SUBORD + N_LEAF
IS_LEAF_FLAT = np.zeros(N_FLAT, dtype=bool)
IS_LEAF_FLAT[FLAT_OFFSET_LEAF:] = True

print(
    f"Taxonomía validada: {N_BASIC} basic, {N_SUBORD} subordinate, "
    f"{N_LEAF} fine; flat={N_FLAT} nodos"
)


## 5. Partición federada y auditoría de cobertura empírica

La partición controla dos fuentes de heterogeneidad:

1. **class skew:** cada clase `fine` se reparte entre clientes mediante una
   Dirichlet con concentración `ALPHA_CLS`;
2. **granularity skew:** cada cliente recibe una distribución sobre las
   profundidades `basic`, `subordinate` y `fine`, muestreada con
   `ALPHA_DEP`.

La función guarda tanto la distribución de profundidades por cliente como su
realización muestra a muestra. Si ya existe una partición, se regenera con la
misma semilla y se exige que coincida exactamente con el fichero guardado.

El recuento de cobertura cuenta, para cada cliente:

- muestras que supervisan la cabeza raíz;
- muestras que supervisan cada cabeza `basic`;
- muestras que supervisan cada cabeza `subordinate`.

Una federación principal solo es válida para el experimento de violación si
todos los nodos están resueltos antes de la intervención.


In [ ]:
def make_partition(partition_seed):
    rng = np.random.default_rng(partition_seed)
    n = len(leaf_local)
    client_of_sample = np.full(n, -1, dtype=np.int64)

    for leaf_id in range(N_LEAF):
        idx = np.where(leaf_local == leaf_id)[0]
        idx = idx.copy()
        rng.shuffle(idx)
        proportions = rng.dirichlet([ALPHA_CLS] * K)
        cuts = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        for client_id, part in enumerate(np.split(idx, cuts)):
            client_of_sample[part] = client_id

    assert np.all(client_of_sample >= 0)

    depth_distribution = rng.dirichlet([ALPHA_DEP] * 3, size=K)
    depth_of_sample = np.empty(n, dtype=np.int64)

    for client_id in range(K):
        idx = np.where(client_of_sample == client_id)[0]
        if len(idx) == 0:
            continue
        depth_of_sample[idx] = rng.choice(
            [1, 2, 3],
            size=len(idx),
            p=depth_distribution[client_id],
        )

    assert np.isin(depth_of_sample, [1, 2, 3]).all()
    return client_of_sample, depth_of_sample, depth_distribution


def partition_path(seed):
    return get_ckpt_dir() / f"partition_seed{seed}.npz"


def load_or_create_partition(seed):
    generated = make_partition(PARTITION_SEED[seed])
    client_gen, depth_gen, nu_gen = generated
    path = partition_path(seed)

    if path.exists():
        saved = np.load(path)
        client_saved = saved["client_of_sample"]
        depth_saved = saved["depth_of_sample"]
        nu_saved = saved["depth_distribution"]

        if not np.array_equal(client_saved, client_gen):
            raise RuntimeError(f"La partición guardada no coincide para seed={seed}")
        if not np.array_equal(depth_saved, depth_gen):
            raise RuntimeError(f"Las profundidades guardadas no coinciden para seed={seed}")
        if not np.allclose(nu_saved, nu_gen):
            raise RuntimeError(f"La Dirichlet guardada no coincide para seed={seed}")
        return client_saved, depth_saved, nu_saved

    np.savez_compressed(
        path,
        client_of_sample=client_gen,
        depth_of_sample=depth_gen,
        depth_distribution=nu_gen,
    )
    return generated


def supervision_certificate(client_of_sample, depth_of_sample):
    root_counts = np.zeros(K, dtype=np.int64)
    basic_counts = np.zeros((K, N_BASIC), dtype=np.int64)
    subord_counts = np.zeros((K, N_SUBORD), dtype=np.int64)

    for k in range(K):
        idx = np.where(client_of_sample == k)[0]
        d = depth_of_sample[idx]
        root_counts[k] = np.sum(d >= 1)

        for b in range(N_BASIC):
            basic_counts[k, b] = np.sum(
                (d >= 2) & (basic_local[idx] == b)
            )
        for s in range(N_SUBORD):
            subord_counts[k, s] = np.sum(
                (d >= 3) & (subord_local[idx] == s)
            )

    resolved_root = root_counts.sum() > 0
    resolved_basic = basic_counts.sum(axis=0) > 0
    resolved_subord = subord_counts.sum(axis=0) > 0

    return {
        "root_counts": root_counts,
        "basic_counts": basic_counts,
        "subord_counts": subord_counts,
        "resolved_root": bool(resolved_root),
        "resolved_basic": resolved_basic,
        "resolved_subord": resolved_subord,
    }


def partition_diagnostics(seed, client_of_sample, depth_of_sample):
    cert = supervision_certificate(client_of_sample, depth_of_sample)
    depth_fraction = {
        str(d): float(np.mean(depth_of_sample == d))
        for d in [1, 2, 3]
    }
    client_sizes = [
        int(np.sum(client_of_sample == k))
        for k in range(K)
    ]
    subord_cohort_sizes = (
        cert["subord_counts"] > 0
    ).sum(axis=0)

    # Un cliente tiene cobertura fine completa solo si observa al menos una
    # etiqueta fine de cada una de las 505 hojas.
    clients_with_complete_fine_coverage = []
    for k in range(K):
        idx = np.where(
            (client_of_sample == k) & (depth_of_sample == 3)
        )[0]
        covered = np.unique(leaf_local[idx])
        if len(covered) == N_LEAF:
            clients_with_complete_fine_coverage.append(k)

    diag = {
        "seed": int(seed),
        "partition_seed": int(PARTITION_SEED[seed]),
        "client_sizes": client_sizes,
        "depth_fraction": depth_fraction,
        "n_unresolved_basic": int((~cert["resolved_basic"]).sum()),
        "n_unresolved_subord": int((~cert["resolved_subord"]).sum()),
        "subord_cohort_size_min": int(subord_cohort_sizes.min()),
        "subord_cohort_size_mean": float(subord_cohort_sizes.mean()),
        "subord_cohort_size_max": int(subord_cohort_sizes.max()),
        "clients_with_complete_fine_coverage": clients_with_complete_fine_coverage,
        "partition_hash": array_sha256(client_of_sample),
        "depth_hash": array_sha256(depth_of_sample),
    }
    atomic_json_dump(diag, get_ckpt_dir() / f"partition_seed{seed}_diagnostics.json")
    return diag, cert


PARTITIONS = {}
for seed in SEEDS:
    client_arr, depth_arr, nu_arr = load_or_create_partition(seed)
    diag, cert = partition_diagnostics(seed, client_arr, depth_arr)

    print(f"\nseed={seed}")
    print("  client sizes:", diag["client_sizes"])
    print("  depth fractions:", diag["depth_fraction"])
    print(
        "  unresolved basic/subordinate:",
        diag["n_unresolved_basic"],
        diag["n_unresolved_subord"],
    )
    print(
        "  subordinate cohort size min/mean/max:",
        diag["subord_cohort_size_min"],
        f"{diag['subord_cohort_size_mean']:.2f}",
        diag["subord_cohort_size_max"],
    )
    print(
        "  clients with complete fine coverage:",
        diag["clients_with_complete_fine_coverage"],
    )

    if diag["n_unresolved_basic"] or diag["n_unresolved_subord"]:
        raise RuntimeError(
            f"La federación seed={seed} no es completamente resoluble. "
            "No debe utilizarse como brazo de control de una violación "
            "localizada sin revisar la partición."
        )
    if diag["clients_with_complete_fine_coverage"]:
        raise RuntimeError(
            f"Hay clientes con cobertura fine completa en seed={seed}: "
            f"{diag['clients_with_complete_fine_coverage']}"
        )

    PARTITIONS[seed] = {
        "client_of_sample": client_arr,
        "depth_of_sample": depth_arr,
        "depth_distribution": nu_arr,
        "diagnostics": diag,
    }


## 6. Modelos y targets

La representación de entrada tiene dimensión `FEATURE_DIM` y permanece fija.
`TreeHead` contiene una cabeza raíz, una cabeza por nodo `basic` y una cabeza
por nodo `subordinate`. `FlatHead` trata los 652 nodos no raíz como un único
alfabeto.

No se añade una proyección compartida entre las features DINO y las cabezas.
Por tanto, en esta versión las cabezas del árbol están desacopladas cuando la
representación se mantiene fija, exactamente en el régimen cubierto por la
reducción teórica de optimización.


In [ ]:
class TreeHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.root = nn.Linear(FEATURE_DIM, N_BASIC)
        self.basic_clf = nn.ModuleList(
            [
                nn.Linear(FEATURE_DIM, int(children_per_basic[b]))
                for b in range(N_BASIC)
            ]
        )
        self.subord_clf = nn.ModuleList(
            [
                nn.Linear(FEATURE_DIM, int(children_per_subord[s]))
                for s in range(N_SUBORD)
            ]
        )


class FlatHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(FEATURE_DIM, N_FLAT)


def flat_target(basic_arr, subord_arr, leaf_arr, depth_arr):
    target = np.empty(len(depth_arr), dtype=np.int64)
    m1 = depth_arr == 1
    m2 = depth_arr == 2
    m3 = depth_arr == 3

    target[m1] = FLAT_OFFSET_BASIC + basic_arr[m1]
    target[m2] = FLAT_OFFSET_SUBORD + subord_arr[m2]
    target[m3] = FLAT_OFFSET_LEAF + leaf_arr[m3]
    return target


def make_initial_state(model_class, init_seed):
    set_all_seeds(init_seed)
    model = model_class()
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


## 7. Pérdida jerárquica exacta

Para un minibatch de tamaño \(B\), cada muestra aporta la suma de las
cross-entropies de las decisiones situadas por encima de su etiqueta observada.
La pérdida del minibatch es la media de esas sumas:

\[
\frac{1}{B}
\left(
\sum_{i:d_i\geq 1}\ell_{\mathrm{root},i}
+
\sum_{i:d_i\geq 2}\ell_{\mathrm{basic},i}
+
\sum_{i:d_i\geq 3}\ell_{\mathrm{subordinate},i}
\right).
\]

La implementación usa `reduction="sum"` en cada grupo y divide una sola vez
por \(B\). Así se evita que un nodo con muy pocos ejemplos en el minibatch
reciba el mismo peso que otro con cientos de ejemplos.


In [ ]:
def batch_features(feature_store, indices):
    # El indexado avanzado del memmap crea solo el minibatch solicitado.
    arr = np.asarray(feature_store[indices], dtype=np.float32)
    return torch.from_numpy(arr).to(device)


def exact_tree_path_loss(model, xb, batch_indices, depth_arr):
    batch_indices = np.asarray(batch_indices, dtype=np.int64)
    d = depth_arr[batch_indices]
    batch_size = len(batch_indices)

    loss_sum = torch.zeros((), dtype=torch.float32, device=device)

    # Decisión root -> basic.
    m1 = d >= 1
    if np.any(m1):
        logits = model.root(xb[m1])
        target = torch.as_tensor(
            T_ROOT_ALL[batch_indices][m1],
            dtype=torch.long,
            device=device,
        )
        loss_sum = loss_sum + F.cross_entropy(
            logits, target, reduction="sum"
        )

    # Decisiones basic -> subordinate.
    m2 = d >= 2
    if np.any(m2):
        for b in np.unique(basic_local[batch_indices][m2]):
            selected = m2 & (basic_local[batch_indices] == b)
            logits = model.basic_clf[int(b)](xb[selected])
            target = torch.as_tensor(
                T_BASIC_ALL[batch_indices][selected],
                dtype=torch.long,
                device=device,
            )
            loss_sum = loss_sum + F.cross_entropy(
                logits, target, reduction="sum"
            )

    # Decisiones subordinate -> fine.
    m3 = d >= 3
    if np.any(m3):
        for s in np.unique(subord_local[batch_indices][m3]):
            selected = m3 & (subord_local[batch_indices] == s)
            logits = model.subord_clf[int(s)](xb[selected])
            target = torch.as_tensor(
                T_SUBORD_ALL[batch_indices][selected],
                dtype=torch.long,
                device=device,
            )
            loss_sum = loss_sum + F.cross_entropy(
                logits, target, reduction="sum"
            )

    return loss_sum / float(batch_size)


## 8. Evaluation

Leaf predictions use global root-to-leaf path products. The saved ancestor-of-decoded-leaf diagnostics are historical metrics, not marginal-prefix accuracy. Flat additionally records the rate of unrestricted predictions at internal nodes.

In [ ]:
@torch.no_grad()
def predict_tree(model, max_samples=None):
    model.eval()
    n_eval = len(Xva) if max_samples is None else min(len(Xva), max_samples)
    predictions = np.empty(n_eval, dtype=np.int64)

    for i0 in range(0, n_eval, 2048):
        i1 = min(i0 + 2048, n_eval)
        idx = np.arange(i0, i1)
        xb = batch_features(Xva, idx)
        n = len(idx)

        root_logp = F.log_softmax(model.root(xb), dim=1)
        leaf_scores = torch.full(
            (n, N_LEAF),
            -torch.inf,
            dtype=torch.float32,
            device=device,
        )

        for b in range(N_BASIC):
            basic_logp = F.log_softmax(
                model.basic_clf[b](xb), dim=1
            )
            for local_s, s in enumerate(subords_by_basic[b]):
                s = int(s)
                fine_logp = F.log_softmax(
                    model.subord_clf[s](xb), dim=1
                )
                leaves = leaves_by_subord[s]
                leaf_scores[:, leaves] = (
                    root_logp[:, b:b+1]
                    + basic_logp[:, local_s:local_s+1]
                    + fine_logp
                )

        predictions[i0:i1] = (
            leaf_scores.argmax(dim=1).cpu().numpy()
        )

    return predictions


def tree_metrics_from_predictions(predictions):
    n = len(predictions)
    true_leaf = val_leaf_local[:n]
    pred_subord = leaf_parent_local[predictions]
    pred_basic = subord_parent[pred_subord]

    leaf_acc = 100.0 * np.mean(predictions == true_leaf)
    subord_acc = 100.0 * np.mean(
        pred_subord == val_subord_local[:n]
    )
    basic_acc = 100.0 * np.mean(
        pred_basic == val_basic_local[:n]
    )
    full_path_acc = 100.0 * np.mean(
        (predictions == true_leaf)
        & (pred_subord == val_subord_local[:n])
        & (pred_basic == val_basic_local[:n])
    )

    return {
        "leaf_acc": float(leaf_acc),
        "subord_acc": float(subord_acc),
        "basic_acc": float(basic_acc),
        "full_path_acc": float(full_path_acc),
    }


@torch.no_grad()
def predict_flat(model, max_samples=None):
    model.eval()
    n_eval = len(Xva) if max_samples is None else min(len(Xva), max_samples)
    leaf_predictions = np.empty(n_eval, dtype=np.int64)
    internal_argmax = np.empty(n_eval, dtype=bool)

    for i0 in range(0, n_eval, 2048):
        i1 = min(i0 + 2048, n_eval)
        idx = np.arange(i0, i1)
        xb = batch_features(Xva, idx)
        logits = model.fc(xb)

        raw_pred = logits.argmax(dim=1).cpu().numpy()
        internal_argmax[i0:i1] = ~IS_LEAF_FLAT[raw_pred]
        leaf_predictions[i0:i1] = (
            logits[:, FLAT_OFFSET_LEAF:].argmax(dim=1).cpu().numpy()
        )

    return leaf_predictions, internal_argmax


def flat_metrics_from_predictions(leaf_predictions, internal_argmax):
    n = len(leaf_predictions)
    pred_subord = leaf_parent_local[leaf_predictions]
    pred_basic = subord_parent[pred_subord]

    return {
        "leaf_acc": float(
            100.0 * np.mean(leaf_predictions == val_leaf_local[:n])
        ),
        "subord_acc": float(
            100.0 * np.mean(pred_subord == val_subord_local[:n])
        ),
        "basic_acc": float(
            100.0 * np.mean(pred_basic == val_basic_local[:n])
        ),
        "raw_internal_node_rate": float(
            100.0 * np.mean(internal_argmax)
        ),
    }


## 9. Entrenamiento auditado de las cabezas en árbol

La misma función implementa `FedAvg-tree` y `ResolvFL`.

- `masked=False`: todos los clientes participan en la media de cada cabeza,
  incluso cuando no han recibido gradiente para ella.
- `masked=True`: cada cabeza se agrega solo entre los clientes que la
  supervisan, ponderados por el número de ejemplos supervisores.

Para comparaciones emparejadas, ambos brazos reciben:

- la misma partición;
- un `state_dict` inicial idéntico;
- la misma semilla de permutación de minibatches.

El estado del generador de NumPy se almacena en cada checkpoint, de modo que
una reanudación continúa exactamente la secuencia de minibatches de esa
ejecución.


In [ ]:
def parameter_supervision_count(
    parameter_key,
    client_indices,
    depth_arr,
):
    d = depth_arr[client_indices]

    if parameter_key.startswith("root."):
        return int(np.sum(d >= 1))

    if parameter_key.startswith("basic_clf."):
        node_id = int(parameter_key.split(".")[1])
        return int(
            np.sum(
                (d >= 2)
                & (basic_local[client_indices] == node_id)
            )
        )

    if parameter_key.startswith("subord_clf."):
        node_id = int(parameter_key.split(".")[1])
        return int(
            np.sum(
                (d >= 3)
                & (subord_local[client_indices] == node_id)
            )
        )

    raise KeyError(f"Parámetro no reconocido: {parameter_key}")


def limit_client_indices_for_quick_test(indices):
    if not QUICK_TEST or len(indices) <= QUICK_MAX_SAMPLES_PER_CLIENT:
        return indices
    # Selección determinista y solo para prueba de humo.
    return indices[:QUICK_MAX_SAMPLES_PER_CLIENT]


def train_tree_audited(
    *,
    masked,
    seed,
    client_of_sample,
    depth_of_sample,
    initial_state,
    order_seed,
    run_tag,
    extra_manifest=None,
):
    rounds = QUICK_ROUNDS if QUICK_TEST else N_ROUNDS
    eval_every = 1 if QUICK_TEST else EVAL_EVERY
    max_val = QUICK_MAX_VAL_SAMPLES if QUICK_TEST else None

    active_ckpt_dir = get_ckpt_dir()
    print(f"  [ruta] {active_ckpt_dir}")
    run_dir = active_ckpt_dir / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / f"tree_seed{seed}.pt"
    log_path = run_dir / f"tree_seed{seed}_log.json"
    pred_path = run_dir / f"tree_seed{seed}_pred.npy"

    initial_hash = state_dict_sha256(initial_state)
    manifest = {
        "experiment_version": EXPERIMENT_VERSION,
        "model": "TreeHead",
        "masked": bool(masked),
        "seed": int(seed),
        "partition_seed": int(PARTITION_SEED[seed]),
        "init_seed": int(INIT_SEED[seed]),
        "order_seed": int(order_seed),
        "partition_hash": array_sha256(client_of_sample),
        "depth_hash": array_sha256(depth_of_sample),
        "initial_state_hash": initial_hash,
        "K": int(K),
        "alpha_cls": float(ALPHA_CLS),
        "alpha_dep": float(ALPHA_DEP),
        "rounds": int(rounds),
        "local_epochs": int(LOCAL_EPOCHS),
        "learning_rate": float(LR),
        "batch_size": int(BATCH),
        "feature_dim": int(FEATURE_DIM),
        "exact_path_loss": True,
        "quick_test": bool(QUICK_TEST),
    }
    if extra_manifest:
        manifest.update(extra_manifest)

    model = TreeHead().to(device)
    model.load_state_dict(initial_state)
    shuffle_rng = np.random.default_rng(order_seed)
    start_round = 0
    log = {
        "rounds": [],
        "leaf_acc": [],
        "subord_acc": [],
        "basic_acc": [],
        "mean_local_path_loss": [],
    }

    if ALLOW_RESUME and ckpt_path.exists():
        state = load_torch_checkpoint(ckpt_path, map_location=device)
        assert_manifest_matches(state["manifest"], manifest)
        model.load_state_dict(state["model"])
        start_round = int(state["round"])
        shuffle_rng.bit_generator.state = state["numpy_rng_state"]
        if log_path.exists():
            with open(log_path, "r", encoding="utf-8") as f:
                log = json.load(f)
        print(
            f"  [checkpoint] {run_tag}, seed={seed}: "
            f"reanuda desde ronda {start_round}"
        )

    if start_round > rounds:
        raise RuntimeError(
            f"Checkpoint en ronda {start_round}, pero el protocolo pide {rounds}"
        )

    client_indices = [
        limit_client_indices_for_quick_test(
            np.where(client_of_sample == k)[0]
        )
        for k in range(K)
    ]

    for rnd in range(start_round, rounds):
        t0 = time.time()
        local_states = []
        round_loss_sum = 0.0
        round_sample_count = 0

        for k in range(K):
            idx_k = client_indices[k]
            if len(idx_k) == 0:
                local_states.append(None)
                continue

            local_model = TreeHead().to(device)
            local_model.load_state_dict(model.state_dict())
            optimizer = torch.optim.SGD(
                local_model.parameters(), lr=LR
            )

            for _ in range(LOCAL_EPOCHS):
                permutation = shuffle_rng.permutation(idx_k)
                for i0 in range(0, len(permutation), BATCH):
                    batch_idx = permutation[i0:i0 + BATCH]
                    xb = batch_features(Xtr, batch_idx)
                    loss = exact_tree_path_loss(
                        local_model,
                        xb,
                        batch_idx,
                        depth_of_sample,
                    )
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    round_loss_sum += float(loss.item()) * len(batch_idx)
                    round_sample_count += len(batch_idx)

            local_states.append(
                {
                    key: value.detach().cpu().clone()
                    for key, value in local_model.state_dict().items()
                }
            )
            del local_model

        current_cpu = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        aggregated = {}

        for key in current_cpu:
            tensors = []
            weights = []

            for k in range(K):
                if local_states[k] is None:
                    continue

                idx_k = client_indices[k]
                if masked:
                    weight = parameter_supervision_count(
                        key, idx_k, depth_of_sample
                    )
                    if weight == 0:
                        continue
                else:
                    weight = len(idx_k)

                tensors.append(local_states[k][key])
                weights.append(float(weight))

            if not tensors:
                aggregated[key] = current_cpu[key]
                continue

            weight_tensor = torch.tensor(weights, dtype=torch.float32)
            weight_tensor = weight_tensor / weight_tensor.sum()
            stacked = torch.stack(tensors, dim=0)
            shape = (-1,) + (1,) * (stacked.dim() - 1)
            aggregated[key] = (
                stacked * weight_tensor.view(shape)
            ).sum(dim=0)

        model.load_state_dict(aggregated)
        mean_local_loss = (
            round_loss_sum / round_sample_count
            if round_sample_count > 0
            else float("nan")
        )

        should_evaluate = (
            (rnd + 1) % eval_every == 0
            or (rnd + 1) == rounds
        )
        if should_evaluate:
            predictions = predict_tree(model, max_samples=max_val)
            metrics = tree_metrics_from_predictions(predictions)

            log["rounds"].append(int(rnd + 1))
            log["leaf_acc"].append(metrics["leaf_acc"])
            log["subord_acc"].append(metrics["subord_acc"])
            log["basic_acc"].append(metrics["basic_acc"])
            log["mean_local_path_loss"].append(float(mean_local_loss))

            np.save(pred_path, predictions)
            atomic_json_dump(log, log_path)

            print(
                f"  [{run_tag}, seed={seed}] "
                f"ronda {rnd+1}/{rounds}, "
                f"loss_local={mean_local_loss:.4f}, "
                f"leaf={metrics['leaf_acc']:.2f}%, "
                f"subord={metrics['subord_acc']:.2f}%, "
                f"basic={metrics['basic_acc']:.2f}% "
                f"({time.time()-t0:.1f}s)"
            )
        else:
            print(
                f"  [{run_tag}, seed={seed}] "
                f"ronda {rnd+1}/{rounds}, "
                f"loss_local={mean_local_loss:.4f} "
                f"({time.time()-t0:.1f}s)"
            )

        atomic_torch_save(
            {
                "model": {
                    key: value.detach().cpu()
                    for key, value in model.state_dict().items()
                },
                "round": int(rnd + 1),
                "numpy_rng_state": shuffle_rng.bit_generator.state,
                "manifest": manifest,
            },
            ckpt_path,
        )

    # Garantiza predicciones finales incluso si el checkpoint ya estaba
    # completo y el fichero de predicciones faltaba.
    if pred_path.exists():
        predictions = np.load(pred_path)
    else:
        predictions = predict_tree(model, max_samples=max_val)
        np.save(pred_path, predictions)

    final_metrics = tree_metrics_from_predictions(predictions)
    return model, log, predictions, final_metrics


## 10. Entrenamiento auditado del baseline plano

El baseline plano utiliza el mismo reparto de clientes y el mismo orden
determinista de minibatches. Su arquitectura es distinta, por lo que compartir
la semilla no implica una inicialización parámetro a parámetro comparable con
la cabeza en árbol.

Esta sección solo se necesita para regenerar la tabla principal. La violación
de resolvability se ejecuta únicamente con ResolvFL, porque evalúa una frontera
informacional y no una comparación entre arquitecturas.


In [ ]:
def train_flat_audited(
    *,
    seed,
    client_of_sample,
    depth_of_sample,
    initial_state,
    order_seed,
    run_tag,
):
    rounds = QUICK_ROUNDS if QUICK_TEST else N_ROUNDS
    max_val = QUICK_MAX_VAL_SAMPLES if QUICK_TEST else None

    active_ckpt_dir = get_ckpt_dir()
    print(f"  [ruta] {active_ckpt_dir}")
    run_dir = active_ckpt_dir / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / f"flat_seed{seed}.pt"
    pred_path = run_dir / f"flat_seed{seed}_pred.npz"

    targets = flat_target(
        basic_local, subord_local, leaf_local, depth_of_sample
    )
    manifest = {
        "experiment_version": EXPERIMENT_VERSION,
        "model": "FlatHead",
        "seed": int(seed),
        "partition_seed": int(PARTITION_SEED[seed]),
        "init_seed": int(INIT_SEED[seed]),
        "order_seed": int(order_seed),
        "partition_hash": array_sha256(client_of_sample),
        "depth_hash": array_sha256(depth_of_sample),
        "initial_state_hash": state_dict_sha256(initial_state),
        "K": int(K),
        "alpha_cls": float(ALPHA_CLS),
        "alpha_dep": float(ALPHA_DEP),
        "rounds": int(rounds),
        "local_epochs": int(LOCAL_EPOCHS),
        "learning_rate": float(LR),
        "batch_size": int(BATCH),
        "feature_dim": int(FEATURE_DIM),
        "quick_test": bool(QUICK_TEST),
    }

    model = FlatHead().to(device)
    model.load_state_dict(initial_state)
    shuffle_rng = np.random.default_rng(order_seed)
    start_round = 0

    if ALLOW_RESUME and ckpt_path.exists():
        state = load_torch_checkpoint(ckpt_path, map_location=device)
        assert_manifest_matches(state["manifest"], manifest)
        model.load_state_dict(state["model"])
        start_round = int(state["round"])
        shuffle_rng.bit_generator.state = state["numpy_rng_state"]
        print(
            f"  [checkpoint] {run_tag}, seed={seed}: "
            f"reanuda desde ronda {start_round}"
        )

    client_indices = [
        limit_client_indices_for_quick_test(
            np.where(client_of_sample == k)[0]
        )
        for k in range(K)
    ]

    for rnd in range(start_round, rounds):
        t0 = time.time()
        local_states = []
        client_weights = []

        for k in range(K):
            idx_k = client_indices[k]
            if len(idx_k) == 0:
                continue

            local_model = FlatHead().to(device)
            local_model.load_state_dict(model.state_dict())
            optimizer = torch.optim.SGD(
                local_model.parameters(), lr=LR
            )

            for _ in range(LOCAL_EPOCHS):
                permutation = shuffle_rng.permutation(idx_k)
                for i0 in range(0, len(permutation), BATCH):
                    batch_idx = permutation[i0:i0 + BATCH]
                    xb = batch_features(Xtr, batch_idx)
                    yb = torch.as_tensor(
                        targets[batch_idx],
                        dtype=torch.long,
                        device=device,
                    )
                    loss = F.cross_entropy(local_model.fc(xb), yb)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            local_states.append(
                {
                    key: value.detach().cpu().clone()
                    for key, value in local_model.state_dict().items()
                }
            )
            client_weights.append(float(len(idx_k)))
            del local_model

        weights = torch.tensor(client_weights, dtype=torch.float32)
        weights = weights / weights.sum()
        aggregated = {}

        for key in model.state_dict():
            stacked = torch.stack(
                [state[key] for state in local_states], dim=0
            )
            shape = (-1,) + (1,) * (stacked.dim() - 1)
            aggregated[key] = (
                stacked * weights.view(shape)
            ).sum(dim=0)

        model.load_state_dict(aggregated)

        atomic_torch_save(
            {
                "model": {
                    key: value.detach().cpu()
                    for key, value in model.state_dict().items()
                },
                "round": int(rnd + 1),
                "numpy_rng_state": shuffle_rng.bit_generator.state,
                "manifest": manifest,
            },
            ckpt_path,
        )
        print(
            f"  [{run_tag}, seed={seed}] "
            f"ronda {rnd+1}/{rounds} "
            f"({time.time()-t0:.1f}s)"
        )

    if pred_path.exists():
        saved = np.load(pred_path)
        leaf_predictions = saved["leaf_predictions"]
        internal_argmax = saved["internal_argmax"]
    else:
        leaf_predictions, internal_argmax = predict_flat(
            model, max_samples=max_val
        )
        np.savez_compressed(
            pred_path,
            leaf_predictions=leaf_predictions,
            internal_argmax=internal_argmax,
        )

    metrics = flat_metrics_from_predictions(
        leaf_predictions, internal_argmax
    )
    return model, leaf_predictions, internal_argmax, metrics


## 11. Comparación principal opcional

Esta celda regenera `FedAvg-flat`, `FedAvg-tree` y `ResolvFL` con el protocolo
auditado. Dentro de cada semilla:

- los tres métodos comparten la misma partición;
- `FedAvg-tree` y `ResolvFL` comparten exactamente el mismo `state_dict`
  inicial;
- las dos variantes de árbol comparten la misma secuencia de minibatches.

Los resultados se guardan en un único JSON canónico. El código no afirma que
la inicialización del baseline plano sea idéntica a la del árbol, porque las
arquitecturas tienen espacios de parámetros distintos.


In [ ]:
MAIN_RESULTS = {}

if RUN_MAIN_EXPERIMENT:
    for seed in SEEDS:
        print(f"\n========== MAIN, SEED {seed} ==========")
        partition = PARTITIONS[seed]
        client_arr = partition["client_of_sample"]
        depth_arr = partition["depth_of_sample"]

        tree_initial = make_initial_state(
            TreeHead, INIT_SEED[seed]
        )
        flat_initial = make_initial_state(
            FlatHead, INIT_SEED[seed]
        )

        _, flat_pred, flat_internal, flat_metrics = train_flat_audited(
            seed=seed,
            client_of_sample=client_arr,
            depth_of_sample=depth_arr,
            initial_state=flat_initial,
            order_seed=ORDER_SEED[seed],
            run_tag="main_flat",
        )

        _, _, unmasked_pred, unmasked_metrics = train_tree_audited(
            masked=False,
            seed=seed,
            client_of_sample=client_arr,
            depth_of_sample=depth_arr,
            initial_state=tree_initial,
            order_seed=ORDER_SEED[seed],
            run_tag="main_unmasked",
            extra_manifest={"condition": "main_unmasked"},
        )

        _, _, masked_pred, masked_metrics = train_tree_audited(
            masked=True,
            seed=seed,
            client_of_sample=client_arr,
            depth_of_sample=depth_arr,
            initial_state=tree_initial,
            order_seed=ORDER_SEED[seed],
            run_tag="main_masked",
            extra_manifest={"condition": "main_masked"},
        )

        MAIN_RESULTS[str(seed)] = {
            "flat": flat_metrics,
            "unmasked": unmasked_metrics,
            "masked": masked_metrics,
            "delta_mask_leaf_acc": float(
                masked_metrics["leaf_acc"]
                - unmasked_metrics["leaf_acc"]
            ),
        }
        atomic_json_dump(
            MAIN_RESULTS,
            get_ckpt_dir() / "imagenet3lf_main_results_audited.json",
        )

    print("\n=== MAIN: RESUMEN ===")
    for method in ["flat", "unmasked", "masked"]:
        values = [
            MAIN_RESULTS[str(seed)][method]["leaf_acc"]
            for seed in SEEDS
        ]
        mean, std = mean_std(values)
        print(f"{method:>10}: leaf={mean:.2f} +/- {std:.2f}")

    flat_internal_values = [
        MAIN_RESULTS[str(seed)]["flat"]["raw_internal_node_rate"]
        for seed in SEEDS
    ]
    mean, std = mean_std(flat_internal_values)
    print(f"flat raw internal-node rate={mean:.2f} +/- {std:.2f}")

    deltas = [
        MAIN_RESULTS[str(seed)]["delta_mask_leaf_acc"]
        for seed in SEEDS
    ]
    mean, std = mean_std(deltas)
    print("delta_mask per seed:", [round(x, 4) for x in deltas])
    print(f"delta_mask={mean:+.2f} +/- {std:.2f}")
else:
    print("RUN_MAIN_EXPERIMENT=False: no se regenera la tabla principal.")


# 12. Experimento de violación de resolvability

## 12.1. Selección determinista del nodo

La intervención se realiza sobre un nodo `subordinate`, porque permite borrar
la distinción entre sus clases `fine` sin eliminar la supervisión de los dos
niveles superiores.

Para evitar selección por resultados:

1. se buscan nodos con cinco hojas;
2. entre ellos se elige el que tiene soporte de entrenamiento más próximo a
   la mediana del conjunto elegible;
3. los empates se resuelven mediante el índice local del nodo.

Si no existe ningún nodo con cinco hojas, se usa el mismo criterio entre nodos
con 4--8 hojas. `TARGET_SUBORD_OVERRIDE` permite fijar manualmente un índice
solo si existe una razón externa documentada antes de observar resultados.


In [ ]:
def subordinate_support_table():
    rows = []
    for s in range(N_SUBORD):
        leaves = leaves_by_subord[s]
        train_mask = subord_local == s
        val_mask = val_subord_local == s

        train_per_leaf = [
            int(np.sum(leaf_local == leaf_id))
            for leaf_id in leaves
        ]
        val_per_leaf = [
            int(np.sum(val_leaf_local == leaf_id))
            for leaf_id in leaves
        ]

        rows.append(
            {
                "subord_local": int(s),
                "subord_raw": py_scalar(subord_raw_list[s]),
                "basic_local": int(subord_parent[s]),
                "basic_raw": py_scalar(
                    basic_values[int(subord_parent[s])]
                ),
                "n_leaves": int(len(leaves)),
                "train_samples": int(train_mask.sum()),
                "val_samples": int(val_mask.sum()),
                "min_train_per_leaf": int(min(train_per_leaf)),
                "min_val_per_leaf": int(min(val_per_leaf)),
                "leaf_local_ids": [int(x) for x in leaves],
            }
        )
    return rows


def select_target_subordinate(rows):
    if TARGET_SUBORD_OVERRIDE is not None:
        matches = [
            row for row in rows
            if row["subord_local"] == int(TARGET_SUBORD_OVERRIDE)
        ]
        if len(matches) != 1:
            raise ValueError(
                f"TARGET_SUBORD_OVERRIDE={TARGET_SUBORD_OVERRIDE} no existe"
            )
        return matches[0], "manual override"

    preferred = [
        row for row in rows
        if row["n_leaves"] == PREFERRED_TARGET_LEAVES
    ]
    if preferred:
        eligible = preferred
        rule = (
            f"n_leaves={PREFERRED_TARGET_LEAVES}; "
            "soporte train más próximo a la mediana"
        )
    else:
        eligible = [
            row for row in rows
            if FALLBACK_MIN_LEAVES
            <= row["n_leaves"]
            <= FALLBACK_MAX_LEAVES
        ]
        rule = (
            f"{FALLBACK_MIN_LEAVES}<=n_leaves<="
            f"{FALLBACK_MAX_LEAVES}; soporte train más próximo "
            "a la mediana"
        )

    if not eligible:
        raise RuntimeError("No hay nodos elegibles para la intervención")

    median_support = float(
        np.median([row["train_samples"] for row in eligible])
    )
    selected = min(
        eligible,
        key=lambda row: (
            abs(row["train_samples"] - median_support),
            row["subord_local"],
        ),
    )
    return selected, rule


SUBORD_SUPPORT = subordinate_support_table()
TARGET_INFO, TARGET_SELECTION_RULE = select_target_subordinate(
    SUBORD_SUPPORT
)
TARGET_SUBORD = int(TARGET_INFO["subord_local"])
TARGET_BASIC = int(TARGET_INFO["basic_local"])
TARGET_LEAVES = np.asarray(
    TARGET_INFO["leaf_local_ids"], dtype=np.int64
)

selection_record = {
    "selection_rule": TARGET_SELECTION_RULE,
    "selected": TARGET_INFO,
    "all_candidates": SUBORD_SUPPORT,
}
atomic_json_dump(
    selection_record,
    get_ckpt_dir() / "violation_target_selection.json",
)

print("Regla de selección:", TARGET_SELECTION_RULE)
print("Nodo seleccionado:")
print(json.dumps(TARGET_INFO, indent=2, default=str))
print(
    f"Chance entre las {len(TARGET_LEAVES)} hojas del subárbol: "
    f"{100.0 / len(TARGET_LEAVES):.2f}%"
)


## 12.2. Construcción de la intervención y comprobación del certificado

Para todas las muestras cuyo verdadero nodo `subordinate` es el seleccionado,
se aplica:

\[
d_{\mathrm{violated}} = \min(d_{\mathrm{resolvable}}, 2).
\]

Solo las etiquetas originalmente `fine` cambian; pasan a observarse en el
nivel `subordinate`. Las etiquetas `basic` y `subordinate` ya existentes se
mantienen.

La celda exige que:

- el nodo objetivo esté resuelto en el brazo de control;
- quede sin ninguna supervisión `fine` en el brazo violado;
- ningún otro nodo cambie su estado de cobertura observada;
- la cantidad de supervisión del prefijo `basic -> subordinate` permanezca
  exactamente igual.


In [ ]:
VIOLATION_DEPTHS = {}
VIOLATION_CERTIFICATES = {}

for seed in SEEDS:
    client_arr = PARTITIONS[seed]["client_of_sample"]
    depth_resolvable = PARTITIONS[seed]["depth_of_sample"].copy()
    target_train_mask = subord_local == TARGET_SUBORD

    depth_violated = depth_resolvable.copy()
    depth_violated[target_train_mask] = np.minimum(
        depth_violated[target_train_mask], 2
    )

    cert_res = supervision_certificate(
        client_arr, depth_resolvable
    )
    cert_vio = supervision_certificate(
        client_arr, depth_violated
    )

    assert cert_res["resolved_subord"][TARGET_SUBORD]
    assert not cert_vio["resolved_subord"][TARGET_SUBORD]

    changed_subord = np.where(
        cert_res["resolved_subord"]
        != cert_vio["resolved_subord"]
    )[0]
    assert np.array_equal(
        changed_subord,
        np.array([TARGET_SUBORD], dtype=np.int64),
    ), (
        "La intervención ha cambiado la resolvability de nodos "
        f"adicionales: {changed_subord.tolist()}"
    )

    # La supervisión de todas las cabezas basic permanece idéntica.
    assert np.array_equal(
        cert_res["basic_counts"], cert_vio["basic_counts"]
    )

    # Ninguna muestra fuera del subárbol cambia de profundidad.
    assert np.array_equal(
        depth_resolvable[~target_train_mask],
        depth_violated[~target_train_mask],
    )

    changed_samples = int(
        np.sum(depth_resolvable != depth_violated)
    )
    original_fine_target = int(
        np.sum(target_train_mask & (depth_resolvable == 3))
    )
    assert changed_samples == original_fine_target

    VIOLATION_DEPTHS[seed] = {
        "resolvable": depth_resolvable,
        "violated": depth_violated,
    }
    VIOLATION_CERTIFICATES[seed] = {
        "changed_samples": changed_samples,
        "target_supervision_resolvable": int(
            cert_res["subord_counts"][:, TARGET_SUBORD].sum()
        ),
        "target_supervision_violated": int(
            cert_vio["subord_counts"][:, TARGET_SUBORD].sum()
        ),
        "changed_resolved_subord_nodes": [
            int(x) for x in changed_subord
        ],
    }

    np.savez_compressed(
        get_ckpt_dir() / f"violation_depths_seed{seed}.npz",
        depth_resolvable=depth_resolvable,
        depth_violated=depth_violated,
    )

    print(f"\nseed={seed}")
    print("  fine labels removed in target subtree:", changed_samples)
    print(
        "  target supervision count:",
        VIOLATION_CERTIFICATES[seed][
            "target_supervision_resolvable"
        ],
        "->",
        VIOLATION_CERTIFICATES[seed][
            "target_supervision_violated"
        ],
    )

atomic_json_dump(
    {
        str(seed): value
        for seed, value in VIOLATION_CERTIFICATES.items()
    },
    get_ckpt_dir() / "violation_certificate_summary.json",
)


## 12.3. Intervention metrics

Save target leaf accuracy, outside leaf accuracy, global leaf accuracy, and the ancestor of each decoded leaf. The last diagnostic depends on the complete leaf decoder and is not marginal-prefix accuracy. Later marginal-prefix evaluations are maintained separately from these historical outputs.

In [ ]:
def violation_metrics(predictions, target_subord):
    n = len(predictions)
    true_leaf = val_leaf_local[:n]
    true_subord = val_subord_local[:n]
    target_mask = true_subord == target_subord
    outside_mask = ~target_mask

    if target_mask.sum() == 0:
        raise RuntimeError(
            "El nodo objetivo no tiene muestras de validación"
        )

    predicted_subord = leaf_parent_local[predictions]
    predicted_basic = subord_parent[predicted_subord]

    metrics = {
        "target_leaf_acc": float(
            100.0
            * np.mean(
                predictions[target_mask] == true_leaf[target_mask]
            )
        ),
        "outside_leaf_acc": float(
            100.0
            * np.mean(
                predictions[outside_mask] == true_leaf[outside_mask]
            )
        ),
        "target_prefix_subord_acc": float(
            100.0
            * np.mean(
                predicted_subord[target_mask] == target_subord
            )
        ),
        "target_prefix_basic_acc": float(
            100.0
            * np.mean(
                predicted_basic[target_mask] == TARGET_BASIC
            )
        ),
        "global_leaf_acc": float(
            100.0 * np.mean(predictions == true_leaf)
        ),
        "n_target_val": int(target_mask.sum()),
        "n_outside_val": int(outside_mask.sum()),
    }
    return metrics


## 12.4. Ejecución emparejada

Para cada semilla se crea una única inicialización de `TreeHead`, que se carga
sin modificaciones en los dos brazos. Ambos utilizan la misma partición y la
misma secuencia de minibatches. La única diferencia experimental es la
profundidad observada de las muestras `fine` del subárbol objetivo.

Se entrenan dos modelos ResolvFL:

- `resolvable`: partición original con cobertura completa observada;
- `violated`: misma partición, con la supervisión `fine` del nodo objetivo
  censurada al nivel `subordinate`.

Los checkpoints se guardan en directorios separados y contienen el
identificador del nodo, los hashes de las profundidades y el manifiesto
completo.


In [ ]:
VIOLATION_RESULTS = {}

if RUN_RESOLVABILITY_VIOLATION:
    for seed in SEEDS:
        print(f"\n========== VIOLATION, SEED {seed} ==========")
        partition = PARTITIONS[seed]
        client_arr = partition["client_of_sample"]

        initial_state = make_initial_state(
            TreeHead, INIT_SEED[seed]
        )

        depth_res = VIOLATION_DEPTHS[seed]["resolvable"]
        _, _, pred_res, metrics_res = train_tree_audited(
            masked=True,
            seed=seed,
            client_of_sample=client_arr,
            depth_of_sample=depth_res,
            initial_state=initial_state,
            order_seed=ORDER_SEED[seed],
            run_tag=(
                f"violation_subord{TARGET_SUBORD}_resolvable"
            ),
            extra_manifest={
                "condition": "resolvable",
                "target_subord": int(TARGET_SUBORD),
                "target_basic": int(TARGET_BASIC),
                "target_n_leaves": int(len(TARGET_LEAVES)),
            },
        )
        violation_res = violation_metrics(
            pred_res, TARGET_SUBORD
        )

        depth_vio = VIOLATION_DEPTHS[seed]["violated"]
        _, _, pred_vio, metrics_vio = train_tree_audited(
            masked=True,
            seed=seed,
            client_of_sample=client_arr,
            depth_of_sample=depth_vio,
            initial_state=initial_state,
            order_seed=ORDER_SEED[seed],
            run_tag=(
                f"violation_subord{TARGET_SUBORD}_violated"
            ),
            extra_manifest={
                "condition": "violated",
                "target_subord": int(TARGET_SUBORD),
                "target_basic": int(TARGET_BASIC),
                "target_n_leaves": int(len(TARGET_LEAVES)),
            },
        )
        violation_vio = violation_metrics(
            pred_vio, TARGET_SUBORD
        )

        deltas = {
            key: float(violation_vio[key] - violation_res[key])
            for key in [
                "target_leaf_acc",
                "outside_leaf_acc",
                "target_prefix_subord_acc",
                "target_prefix_basic_acc",
                "global_leaf_acc",
            ]
        }

        VIOLATION_RESULTS[str(seed)] = {
            "resolvable": violation_res,
            "violated": violation_vio,
            "violated_minus_resolvable": deltas,
            "certificate": VIOLATION_CERTIFICATES[seed],
        }
        atomic_json_dump(
            VIOLATION_RESULTS,
            get_ckpt_dir() / "imagenet3lf_violation_results.json",
        )

        print("\n  FINAL PAIR")
        print(
            "  target leaves:",
            f"{violation_res['target_leaf_acc']:.2f}",
            "->",
            f"{violation_vio['target_leaf_acc']:.2f}",
        )
        print(
            "  other leaves:",
            f"{violation_res['outside_leaf_acc']:.2f}",
            "->",
            f"{violation_vio['outside_leaf_acc']:.2f}",
        )
        print(
            "  subordinate prefix:",
            f"{violation_res['target_prefix_subord_acc']:.2f}",
            "->",
            f"{violation_vio['target_prefix_subord_acc']:.2f}",
        )
else:
    print(
        "RUN_RESOLVABILITY_VIOLATION=False: "
        "no se ejecuta la intervención."
    )


## 12.5. Saved intervention summary

Report descriptive means, sample standard deviations and individual paired differences (`violated - resolvable`). Historical ancestor-of-decoded-leaf fields are not eligible as marginal-prefix results.

In [ ]:
if RUN_RESOLVABILITY_VIOLATION:
    metric_keys = [
        "target_leaf_acc",
        "outside_leaf_acc",
        "target_prefix_subord_acc",
        "target_prefix_basic_acc",
        "global_leaf_acc",
    ]

    canonical = {
        "experiment_version": EXPERIMENT_VERSION,
        "target": TARGET_INFO,
        "selection_rule": TARGET_SELECTION_RULE,
        "n_seeds": len(SEEDS),
        "seeds": [int(s) for s in SEEDS],
        "metrics": {},
    }

    print("\n=== IMAGENET-3L/F RESOLVABILITY VIOLATION ===")
    print(
        f"target subordinate local={TARGET_SUBORD}, "
        f"raw={TARGET_INFO['subord_raw']}, "
        f"leaves={len(TARGET_LEAVES)}"
    )

    for key in metric_keys:
        res_values = [
            VIOLATION_RESULTS[str(seed)]["resolvable"][key]
            for seed in SEEDS
        ]
        vio_values = [
            VIOLATION_RESULTS[str(seed)]["violated"][key]
            for seed in SEEDS
        ]
        deltas = [
            vio - res
            for res, vio in zip(res_values, vio_values)
        ]

        res_mean, res_std = mean_std(res_values)
        vio_mean, vio_std = mean_std(vio_values)
        delta_mean, delta_std = mean_std(deltas)

        canonical["metrics"][key] = {
            "resolvable_per_seed": res_values,
            "violated_per_seed": vio_values,
            "delta_per_seed": deltas,
            "resolvable_mean": res_mean,
            "resolvable_std": res_std,
            "violated_mean": vio_mean,
            "violated_std": vio_std,
            "delta_mean": delta_mean,
            "delta_std": delta_std,
        }

        print(f"\n{key}")
        print(f"  resolvable: {res_mean:.2f} +/- {res_std:.2f}")
        print(f"  violated:   {vio_mean:.2f} +/- {vio_std:.2f}")
        print(
            "  paired deltas:",
            [round(value, 4) for value in deltas],
        )
        print(
            f"  delta:      {delta_mean:+.2f} +/- "
            f"{delta_std:.2f}"
        )

    atomic_json_dump(
        canonical,
        get_ckpt_dir() / "imagenet3lf_violation_canonical_summary.json",
    )
    print(
        "\nResumen guardado en:",
        get_ckpt_dir() / "imagenet3lf_violation_canonical_summary.json",
    )


## 13. Run verification

Check protocol version, full 150-round completion, paired partition and initialization, checkpoint manifests, and the observed coverage change. This source uses fixed DINO features and linear heads with SGD momentum 0. Saved filenames containing `canonical` do not establish eligibility for manuscript metrics. See `docs/RECOVERED_EXPERIMENTS.md` for the result-to-source mapping and remaining dependencies.